# Prepare prompt-neutral inputs for the bounded task-treatment pilot
Use a GPU. Attach private dataset `thestonedape/task-aware-eegtotext` and checkpoint dataset `thestonedape/glim-zuco-checkpoint`; enable Internet and private secret `GITHUB_TOKEN`. This notebook extracts the same prompt-neutral GLIM EEG vectors for all four pilot configurations and frozen GLIM text targets. It does not train a model and cannot access the held-out test split. The earlier canonical-prompt vector artifact remains untouched factor evidence.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'e09e4c54133af8786dbd16a558c0f31ac00f3874'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
DONOR_SHA256 = '2f7cd3e7ba9713819bdc6ed90d18077997df395b9d05238d63e871d5f58ce75b'
OUTPUT = '/kaggle/working/task-aware-eeg2text-prompt-neutral-pilot-inputs'
EEG_OUTPUT = OUTPUT + '/eeg'
TEXT_OUTPUT = OUTPUT + '/text'
SMOKE_ROOT = '/kaggle/working/prompt-neutral-pilot-input-smoke'
EEG_BATCH_SIZE, EEG_CHUNK_SIZE = 8, 128
TEXT_BATCH_SIZE, TEXT_CHUNK_SIZE = 64, 512
FORCE_FRESH = False
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(len(value) == 64 for value in (EXPECTED_INDEX_SHA256, CHECKPOINT_SHA256, DONOR_SHA256))

In [ ]:
import csv, glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path): shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.4'], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_frozen_vector_contract.py')], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_frozen_text_vector_contract.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'project_adapters.test_glim_representation', 'project_adapters.test_task_treatment_pilots', 'evaluation.test_task_treatment_pilot_contract'], check=True, cwd=WORKTREE)
print({'code_regressions': 'PASS', 'project_commit': actual_commit, 'glim_commit': GLIM_COMMIT})

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, ('Attach exactly one GLIM checkpoint', checkpoint_paths)
checkpoint = checkpoint_paths[0]
def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''): state.update(block)
    return state.hexdigest()
assert digest(checkpoint) == CHECKPOINT_SHA256
if FORCE_FRESH and os.path.exists(OUTPUT): shutil.rmtree(OUTPUT)
resume_manifests = glob.glob('/kaggle/input/**/eeg/vector_manifest.json', recursive=True)
resume_roots = sorted({str(Path(path).parents[1]) for path in resume_manifests if json.load(open(path, encoding='utf-8')).get('prompt_mode') == 'all_masked'})
assert len(resume_roots) <= 1, ('Attach at most one prompt-neutral partial output', resume_roots)
if not os.path.exists(OUTPUT) and resume_roots and not FORCE_FRESH:
    print({'resume_from': resume_roots[0]})
    shutil.copytree(resume_roots[0], OUTPUT)
print({'dataset_root': dataset_root, 'checkpoint': checkpoint, 'output_exists': os.path.exists(OUTPUT)})

In [ ]:
def eeg_command(output, batch_size, chunk_size, smoke_limit=None):
    command = [sys.executable, os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_vectors.py'), '--dataset-root', dataset_root, '--output-root', output, '--glim-root', GLIM_WORKTREE, '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT, '--expected-index-sha256', EXPECTED_INDEX_SHA256, '--expected-checkpoint-sha256', CHECKPOINT_SHA256, '--expected-donor-sha256', DONOR_SHA256, '--device', 'cuda', '--batch-size', str(batch_size), '--chunk-size', str(chunk_size), '--prompt-mode', 'all_masked']
    if smoke_limit is not None: command += ['--smoke-limit', str(smoke_limit)]
    return command
def text_command(output, batch_size, chunk_size, smoke_limit=None):
    command = [sys.executable, os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_text_vectors.py'), '--dataset-root', dataset_root, '--output-root', output, '--glim-root', GLIM_WORKTREE, '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT, '--expected-index-sha256', EXPECTED_INDEX_SHA256, '--expected-checkpoint-sha256', CHECKPOINT_SHA256, '--device', 'cuda', '--batch-size', str(batch_size), '--chunk-size', str(chunk_size)]
    if smoke_limit is not None: command += ['--smoke-limit', str(smoke_limit)]
    return command
if os.path.exists(SMOKE_ROOT): shutil.rmtree(SMOKE_ROOT)
os.makedirs(SMOKE_ROOT)
subprocess.run(eeg_command(SMOKE_ROOT + '/eeg', 1, 1, smoke_limit=1), check=True)
subprocess.run(text_command(SMOKE_ROOT + '/text', 1, 1, smoke_limit=1), check=True)
eeg_smoke = json.load(open(SMOKE_ROOT + '/eeg/vector_manifest.json', encoding='utf-8'))
text_smoke = json.load(open(SMOKE_ROOT + '/text/text_vector_manifest.json', encoding='utf-8'))
assert eeg_smoke['prompt_mode'] == 'all_masked' and eeg_smoke['checks']['held_out_test_accessed'] is False
assert text_smoke['run_mode'] == 'smoke' and text_smoke['checks']['held_out_test_accessed'] is False
print({'real_prompt_neutral_eeg_smoke': 'PASS', 'real_text_target_smoke': 'PASS', 'text_model_id': text_smoke['text_model_id'], 'text_dtype': text_smoke['text_dtype']})

In [ ]:
# Long resumable EEG cell. Re-run after interruption; hash-valid chunks are reused.
os.makedirs(OUTPUT, exist_ok=True)
subprocess.run(eeg_command(EEG_OUTPUT, EEG_BATCH_SIZE, EEG_CHUNK_SIZE), check=True)

In [ ]:
# Frozen text targets are much smaller because repeated trials share one normalized-text identity.
subprocess.run(text_command(TEXT_OUTPUT, TEXT_BATCH_SIZE, TEXT_CHUNK_SIZE), check=True)

In [ ]:
eeg_manifest_path = EEG_OUTPUT + '/vector_manifest.json'
text_manifest_path = TEXT_OUTPUT + '/text_vector_manifest.json'
eeg_manifest = json.load(open(eeg_manifest_path, encoding='utf-8'))
text_manifest = json.load(open(text_manifest_path, encoding='utf-8'))
expected_counts = {'correct_train': 17908, 'correct_val': 2200, 'matched_wrong_val': 2200, 'zero_val': 2200, 'gaussian_val': 2200}
assert eeg_manifest['status'] == 'pass' and eeg_manifest['prompt_mode'] == 'all_masked'
assert eeg_manifest['condition_counts'] == expected_counts and eeg_manifest['source_index_sha256'] == EXPECTED_INDEX_SHA256
assert eeg_manifest['checkpoint_sha256'] == CHECKPOINT_SHA256 and eeg_manifest['wrong_eeg_donor_sha256'] == DONOR_SHA256
assert eeg_manifest['checks']['held_out_test_accessed'] is False
assert text_manifest['status'] == 'pass' and text_manifest['run_mode'] == 'full_development'
assert text_manifest['mapped_trials'] == 20108 and text_manifest['split_counts'] == {'train': 17908, 'val': 2200}
assert text_manifest['source_index_sha256'] == EXPECTED_INDEX_SHA256 and text_manifest['checkpoint_sha256'] == CHECKPOINT_SHA256
assert text_manifest['checks']['held_out_test_accessed'] is False
for root, manifest in ((EEG_OUTPUT, eeg_manifest), (TEXT_OUTPUT, text_manifest)):
    for chunk in manifest['chunks']:
        if root == EEG_OUTPUT:
            path = os.path.join(root, 'vectors', f"{chunk['condition']}_{chunk['chunk_number']:05d}.npz")
        else:
            path = os.path.join(root, 'vectors', f"text_{chunk['chunk_number']:05d}.npz")
        assert digest(path) == chunk['vector_npz_sha256'], path
with open(EEG_OUTPUT + '/vector_index.csv', encoding='utf-8', newline='') as handle: eeg_index = list(csv.DictReader(handle))
with open(TEXT_OUTPUT + '/trial_text_targets.csv', encoding='utf-8', newline='') as handle: text_mapping = list(csv.DictReader(handle))
assert len(eeg_index) == 26708 and {row['prompt_mode'] for row in eeg_index} == {'all_masked'}
assert all(row['phase'] != 'test' for row in eeg_index) and all(row['split'] != 'test' for row in text_mapping)
combined = {'status': 'pass', 'schema_version': 1, 'project_commit': actual_commit, 'glim_commit': GLIM_COMMIT, 'source_index_sha256': EXPECTED_INDEX_SHA256, 'checkpoint_sha256': CHECKPOINT_SHA256, 'pilot_contract_sha256': digest(WORKTREE + '/evaluation/task_treatment_pilot_contract.json'), 'eeg_prompt_mode': 'all_masked', 'eeg_manifest_sha256': digest(eeg_manifest_path), 'text_manifest_sha256': digest(text_manifest_path), 'eeg_vector_index_sha256': eeg_manifest['vector_index_sha256'], 'text_vector_index_sha256': text_manifest['text_vector_index_sha256'], 'trial_text_targets_sha256': text_manifest['trial_text_targets_sha256'], 'held_out_test_accessed': False}
with open(OUTPUT + '/pilot_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(combined, handle, indent=2, sort_keys=True); handle.write('\n')
run_metadata = {'status': 'pass', 'project_commit': actual_commit, 'glim_commit': GLIM_COMMIT, 'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'transformers': __import__('transformers').__version__, 'test_accessed': False}
with open(OUTPUT + '/run_metadata.json', 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True); handle.write('\n')
summary = {'eeg_rows': len(eeg_index), 'eeg_chunks': len(eeg_manifest['chunks']), 'unique_text_targets': text_manifest['unique_text_identities'], 'mapped_trials': len(text_mapping), 'pilot_input_manifest_sha256': digest(OUTPUT + '/pilot_input_manifest.json')}
for path in (WORKTREE, GLIM_WORKTREE, SMOKE_ROOT):
    if os.path.exists(path): shutil.rmtree(path)
print(summary)
print('PROMPT-NEUTRAL PILOT INPUT PREPARATION: PASS')